# SO-app v0.1 en Kaggle — todo en una celda

Ubuntu 22.04 nativo con `proot` + Chrome real + OpenCode web + túnel Cloudflare.
Corre la única celda de código y espera el `TUNEL: ...` al final.
Primera vez tarda 10-15 min (~300MB de descargas). Re-correr reusa todo.
Requiere Internet activado en el notebook.

In [ ]:
%%bash
set -u
export LC_ALL=C LANG=C
DIR=/kaggle/working/vmdata/proot
mkdir -p "$DIR"
R() { proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -b "$DIR/rootfs/etc/resolv.conf:/etc/resolv.conf" -w / "$@"; }

echo "=== [1/5] proot ==="
command -v proot >/dev/null || { echo "instalando proot..."; apt-get update && apt-get install -y proot; }
proot --version | head -1

echo "=== [2/5] rootfs ==="
if [ ! -d "$DIR/rootfs/bin" ]; then
  echo "descargando ubuntu-base (~29MB)..."
  curl -fSL --retry 3 -o "$DIR/ubuntu-base.tar.gz" https://cdimage.ubuntu.com/ubuntu-base/releases/22.04/release/ubuntu-base-22.04-base-amd64.tar.gz
  mkdir -p "$DIR/rootfs"
  echo "extrayendo..."
  tar -xzf "$DIR/ubuntu-base.tar.gz" -C "$DIR/rootfs"
  echo "rootfs OK"
else echo "rootfs existente, reuso"; fi
rm -f "$DIR/rootfs/etc/resolv.conf"
printf 'nameserver 8.8.8.8\nnameserver 1.1.1.1\n' > "$DIR/rootfs/etc/resolv.conf"
R /bin/sh -c 'dpkg --configure -a 2>&1 | tail -1 || true'
if R /bin/sh -c 'command -v curl' >/dev/null 2>&1; then echo "curl existente, reuso";
else
  echo "apt update + curl (1-2 min)..."
  R /bin/sh -c 'apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true update && apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -y curl ca-certificates'
fi
R /bin/sh -c 'command -v curl && echo CURL_OK' || { echo "ERROR: sin curl"; exit 1; }

echo "=== [3/5] Chrome ==="
if R /bin/sh -c 'LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu /opt/google/chrome/chrome --version --no-sandbox' 2>/dev/null; then echo "Chrome existente, reuso";
else
  [ -s /tmp/chrome.deb ] || R /bin/sh -c 'curl -fSL --retry 2 -o /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && ls -lh /tmp/chrome.deb'
  echo "desempaquetando Chrome (ignora errores de postinst)..."
  R /bin/sh -c 'dpkg -i --force-all /tmp/chrome.deb 2>&1 | tail -1 || true'
  echo "trayendo librerias..."
  R /bin/sh -c 'apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true update -qq; apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -f -y 2>&1 | tail -1 || true'
  R /bin/sh -c 'LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu /opt/google/chrome/chrome --version --no-sandbox' || { echo "ERROR: Chrome no corre"; exit 1; }
fi

echo "=== [4/5] opencode web puerto 4096 ==="
OCBIN=""; for c in /tools/node/bin/opencode "$(command -v opencode 2>/dev/null)"; do [ -n "$c" ] && [ -x "$c" ] && OCBIN="$c" && break; done
if [ -z "$OCBIN" ]; then echo "instalando opencode (unos min)..."; npm install -g opencode-ai | tail -2; OCBIN="$(command -v opencode)"; fi
[ -x "$OCBIN" ] || { echo "ERROR: sin binario opencode"; exit 1; }
echo "opencode: $OCBIN"
pkill -f "opencode web --hostname 127.0.0.1 --port 4096" 2>/dev/null || true; sleep 1
cd /kaggle/working && nohup "$OCBIN" web --hostname 127.0.0.1 --port 4096 > /tmp/opencode-web.log 2>&1 &
for i in $(seq 1 30); do (echo > /dev/tcp/127.0.0.1/4096) 2>/dev/null && break; sleep 2; done
(echo > /dev/tcp/127.0.0.1/4096) 2>/dev/null && echo "OPENCODE_UP" || { echo "ERROR: opencode no levanto:"; tail -10 /tmp/opencode-web.log; exit 1; }

echo "=== [5/5] tunel (hasta ~2 min) ==="
if [ ! -x /tmp/cloudflared ]; then echo "descargando cloudflared..."; curl -fSL --retry 2 -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared && echo "cloudflared OK"; else echo "cloudflared existente, reuso"; fi
pkill -f "cloudflared tunnel --url http://127.0.0.1:4096" 2>/dev/null || true; sleep 1
nohup /tmp/cloudflared tunnel --url http://127.0.0.1:4096 > /tmp/tunnel.log 2>&1 &
URL=""
for i in $(seq 1 45); do URL=$(grep -oE "https://[a-zA-Z0-9.-]+\\.trycloudflare\\.com" /tmp/tunnel.log 2>/dev/null | head -1); [ -n "$URL" ] && break; sleep 2; done
[ -n "$URL" ] && echo "TUNEL: $URL" || { echo "ERROR: sin URL, log:"; tail -15 /tmp/tunnel.log; exit 1; }
echo "Listo. Los servicios siguen corriendo aunque la celda termine. El link es publico: no lo compartas."